## Eye Detection Server


In [1]:
import os
from flask import Flask, request, jsonify
from tensorflow.lite.python.interpreter import Interpreter
import cv2
import numpy as np
from werkzeug.utils import secure_filename
from io import BytesIO

# Initialize Flask app
app = Flask(__name__)

# Directory to save uploaded images
UPLOAD_FOLDER = 'uploaded_eyes'
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

# Load the pre-trained TFLite model for eye detection
saved_model_path = 'mobileNet_eye_model.tflite'
interpreter = Interpreter(model_path=saved_model_path)
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Function to preprocess the image for the model
def preprocess_image(image, input_size):
    # Resize to the input size of the model
    resized_image = cv2.resize(image, (input_size[1], input_size[2]))

    # Normalize pixel values to the range [0, 1]
    normalized_image = resized_image.astype(np.float32) / 255.0

    # Add batch dimension
    input_tensor = np.expand_dims(normalized_image, axis=0)
    return input_tensor

# Function to predict the eye state
def predict_eye_state(image, interpreter):
    # Preprocess the eye image
    input_size = input_details[0]['shape']  # Model input size (batch, height, width, channels)
    input_tensor = preprocess_image(image, input_size)

    # Set the input tensor
    interpreter.set_tensor(input_details[0]['index'], input_tensor)

    # Run inference
    interpreter.invoke()

    # Get the output tensor
    output_data = interpreter.get_tensor(output_details[0]['index'])[0]

    # Interpret the prediction
    return "eye open" if output_data > 0.5 else "eye closed"
@app.route('/status', methods=['GET'])
def status():
    try:
        return jsonify({'status': 'Server is running'}), 200
    except Exception as e:
        import traceback
        return jsonify({'error': f"An error occurred: {traceback.format_exc()}"}), 500
    
# Endpoint for handling eye images and performing inference
@app.route('/upload_eye_images', methods=['POST'])
def upload_eye_images():
    if 'leftEye' not in request.files or 'rightEye' not in request.files:
        return jsonify({'error': 'No eye images found in the request'}), 400

    try:
        # Retrieve the eye images
        left_eye_file = request.files['leftEye']
        right_eye_file = request.files['rightEye']

        # Save the images to the server
        left_eye_filename = secure_filename(left_eye_file.filename)
        right_eye_filename = secure_filename(right_eye_file.filename)

        left_eye_path = os.path.join(UPLOAD_FOLDER, left_eye_filename)
        right_eye_path = os.path.join(UPLOAD_FOLDER, right_eye_filename)

        left_eye_file.save(left_eye_path)
        right_eye_file.save(right_eye_path)

        # Read the eye images and perform inference
        left_eye_image = cv2.imread(left_eye_path)
        right_eye_image = cv2.imread(right_eye_path)

        left_eye_status = predict_eye_state(left_eye_image, interpreter)
        right_eye_status = predict_eye_state(right_eye_image, interpreter)

        # Return the results
        return jsonify({
            'leftEyeStatus': left_eye_status,
            'rightEyeStatus': right_eye_status,
            # 'savedFiles': {
            #     'leftEye': left_eye_path,
            #     'rightEye': right_eye_path
            # }
        }), 200

    except Exception as e:
        import traceback
        return jsonify({'error': f"An error occurred: {traceback.format_exc()}"}), 500

# Run the Flask app
if __name__ == '__main__':
    app.run(host='0.0.0.0', debug=False, use_reloader=False, port=8080)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8080
 * Running on http://192.168.18.217:8080
Press CTRL+C to quit
192.168.18.104 - - [11/Dec/2024 00:16:23] "POST /upload_eye_images HTTP/1.1" 200 -
192.168.18.104 - - [11/Dec/2024 00:18:34] "POST /upload_eye_images HTTP/1.1" 200 -
192.168.18.104 - - [11/Dec/2024 00:18:36] "POST /upload_eye_images HTTP/1.1" 200 -
192.168.18.104 - - [11/Dec/2024 00:18:36] "POST /upload_eye_images HTTP/1.1" 200 -
192.168.18.104 - - [11/Dec/2024 00:18:37] "POST /upload_eye_images HTTP/1.1" 200 -
192.168.18.104 - - [11/Dec/2024 00:18:37] "POST /upload_eye_images HTTP/1.1" 200 -
192.168.18.104 - - [11/Dec/2024 00:18:37] "POST /upload_eye_images HTTP/1.1" 200 -
192.168.18.104 - - [11/Dec/2024 00:18:38] "POST /upload_eye_images HTTP/1.1" 200 -
192.168.18.104 - - [11/Dec/2024 00:18:38] "POST /upload_eye_images HTTP/1.1" 200 -
192.168.18.104 - - [11/Dec/2024 00:18:39] "POST /upload_eye_images HTTP/1.1" 200 -
192.168.18.104 - - [11/Dec/2024 00